<a href="https://colab.research.google.com/github/tiendungnguyenw0907-boop/DocumentIntelligence/blob/main/_notebooks/2026-06-25-ReadPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Layer 1. Physical Understanding**

## 1. Ingestion

> *Tiếp nhận tài liệu từ các nguồn khác nhau và đánh giá chất lượng đầu vào nhằm bảo đảm tài liệu đủ điều kiện để đưa vào quy trình xử lý tự động. Đây là bước kiểm soát chất lượng đầu vào của toàn bộ hệ thống*



### 1.1 Tiếp nhận tài liệu

### 1.2 Kiểm tra chất lượng

# 2. Physical Analysis

> *Phân tích các đặc tính vật lý và kỹ thuật của tài liệu nhằm hiểu tài liệu được tạo ra như thế nào, gồm những thành phần gì và cần áp dụng chiến lược xử lý nào. Kết quả của bước này giúp hệ thống xác định phương pháp OCR, nhận dạng bố cục và trích xuất thông tin phù hợp*



## 2.1. PDF Parsing & Format Analysis
> **Mục tiêu**
> *   *Hiểu bản chất vật lý của tài liệu trước khi xử lý*
> *   *Xác định loại PDF và chiến lược xử lý phù hợp*
> *   *Phân tích cấu trúc kỹ thuật của tài liệu*

> **Bài toán cần giải quyết**
>   Tài liệu không chỉ có văn bản
>   Một trang có thể chứa (*Tiêu đề, Đoạn văn, Bảng biểu, Hình ảnh, Header, Footer, Chú thích*)

> **Do đó**
>*   *OCR toàn trang dễ làm mất ngữ cảnh*
>*   *Không xác định được ranh giới các đối tượng*
>*   *Khó phục hồi cấu trúc tài liệu sau OCR*


### 2.1.1. Phân tích cấu trúc PDF

#### 2.1.1.1 Installation

In [24]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 71.2 MB/s eta 0:00:00


#### 2.1.1.2. UPLOAD PDF




In [25]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("PDF:", pdf_file)

Saving SachBCTHKQKT2021.pdf to SachBCTHKQKT2021.pdf
PDF: SachBCTHKQKT2021.pdf


#### 2.1.1.3. PDF Structure Analyzer

In [26]:
import fitz
import json
import os

from tqdm import tqdm
from collections import Counter


class PDFStructureAnalyzer:

    def __init__(self, pdf_path):

        self.pdf_path = pdf_path

        self.doc = fitz.open(pdf_path)

    def analyze(self):

        file_size_mb = round(
            os.path.getsize(self.pdf_path) / 1024 / 1024,
            2
        )

        result = {

            "document_id": "DOC001",

            "file_info": {},

            "document_structure": {},

            "object_statistics": {},

            "font_statistics": [],

            "page_structure": [],

            "page_summary": {},

            "technical_flags": {}

        }

        result["file_info"] = {

            "file_name": os.path.basename(self.pdf_path),

            "file_size_mb": file_size_mb,

            "pdf_version": "Unknown",

            "encrypted": self.doc.needs_pass,

            "page_count": len(self.doc)

        }

        total_text_objects = 0

        total_image_objects = 0

        total_annotation_objects = 0

        total_vector_objects = 0

        font_counter = Counter()

        pages_with_text = 0

        pages_without_text = 0

        pages_with_images = 0

        pages_with_annotations = 0

        rotated_pages = 0

        has_bookmark = False

        has_hyperlink = False

        has_text_layer = False

        has_image_layer = False

        page_structure = []

        print("Analyzing pages...")

        for page_num in tqdm(range(len(self.doc))):

            page = self.doc[page_num]

            try:

                text = page.get_text()

            except:

                text = ""

            try:

                text_dict = page.get_text("dict")

                blocks = text_dict.get("blocks", [])

                text_object_count = len(blocks)

            except:

                text_object_count = 0

            try:

                image_count = len(page.get_images(full=True))

            except:

                image_count = 0

            try:

                fonts = page.get_fonts()

            except:

                fonts = []

            font_names = []

            for f in fonts:

                if len(f) > 3:

                    font_name = str(f[3])

                    font_names.append(font_name)

                    font_counter[font_name] += 1

            annotation_count = 0

            try:

                annots = page.annots()

                if annots:

                    annotation_count = len(list(annots))

            except:

                pass

            try:

                links = page.get_links()

                if len(links) > 0:

                    has_hyperlink = True

            except:

                pass

            page_info = {

                "page_number": page_num + 1,

                "has_text_layer": len(text.strip()) > 0,

                "text_object_count": text_object_count,

                "image_object_count": image_count,

                "font_count": len(set(font_names)),

                "annotation_count": annotation_count,

                "page_rotation": page.rotation

            }

            page_structure.append(page_info)

            total_text_objects += text_object_count

            total_image_objects += image_count

            total_annotation_objects += annotation_count

            if len(text.strip()) > 0:

                pages_with_text += 1

                has_text_layer = True

            else:

                pages_without_text += 1

            if image_count > 0:

                pages_with_images += 1

                has_image_layer = True

            if annotation_count > 0:

                pages_with_annotations += 1

            if page.rotation != 0:

                rotated_pages += 1

        try:

            toc = self.doc.get_toc()

            has_bookmark = len(toc) > 0

        except:

            pass

        result["document_structure"] = {

            "has_text_layer": has_text_layer,

            "has_image_layer": has_image_layer,

            "has_annotation": pages_with_annotations > 0,

            "has_form_field": False,

            "has_embedded_font": len(font_counter) > 0,

            "has_vector_graphic": False,

            "has_bookmark": has_bookmark,

            "has_hyperlink": has_hyperlink,

            "has_watermark": False

        }

        result["object_statistics"] = {

            "total_text_objects": total_text_objects,

            "total_image_objects": total_image_objects,

            "total_font_objects": len(font_counter),

            "total_annotation_objects": total_annotation_objects,

            "total_vector_objects": total_vector_objects

        }

        font_statistics = []

        for font_name, count in font_counter.most_common():

            font_statistics.append({

                "font_name": font_name,

                "usage_count": count

            })

        result["font_statistics"] = font_statistics

        result["page_structure"] = page_structure

        result["page_summary"] = {

            "pages_with_text_layer": pages_with_text,

            "pages_without_text_layer": pages_without_text,

            "pages_with_images": pages_with_images,

            "pages_with_annotations": pages_with_annotations,

            "rotated_pages": rotated_pages

        }

        result["technical_flags"] = {

            "contains_scanned_pages":
                pages_without_text > 0,

            "contains_native_pages":
                pages_with_text > 0,

            "contains_mixed_pages":
                pages_with_text > 0 and pages_without_text > 0,

            "contains_large_images":
                total_image_objects > 100,

            "contains_rotated_pages":
                rotated_pages > 0

        }

        return result

#### 2.1.1.4. Chạy phân tích

In [27]:
analyzer = PDFStructureAnalyzer(pdf_file)

result = analyzer.analyze()

print("Done")

Analyzing pages...


100%|██████████| 284/284 [00:02<00:00, 110.72it/s]

Done


#### 2.1.1.5. Xuất JSON

In [28]:
from pathlib import Path
OUTPUT_DIR = Path("output")
RENDER_DIR = OUTPUT_DIR / "1.Physical_Analysis" / "1.PDF_Parsing_Format_Analysis"
RENDER_DIR.mkdir(parents=True, exist_ok=True)
output_pdf_structure_profile_file = RENDER_DIR /"1.pdf_structure_profile.json"

with open(
    output_pdf_structure_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", output_pdf_structure_profile_file)

Saved: output/1.Physical_Analysis/1.PDF_Parsing_Format_Analysis/1.pdf_structure_profile.json


#### 2.1.1.6. Download JSON

In [29]:
from google.colab import files

files.download(output_pdf_structure_profile_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 2.1.2. Xác định loại PDF

#### 2.1.2.1 Load PDF Structure Profile

In [30]:
import json

with open(output_pdf_structure_profile_file, "r", encoding="utf-8") as f:
    structure_profile = json.load(f)

print("Loaded")

Loaded


#### 2.1.2.2 PDF Type Analyzer

In [31]:
class PDFTypeAnalyzer:

    def __init__(self, structure_profile):

        self.structure_profile = structure_profile

    def analyze(self):

        page_structure = self.structure_profile["page_structure"]

        total_pages = len(page_structure)

        native_pages = []
        scan_pages = []
        mixed_pages = []

        for page in page_structure:

            has_text = page["has_text_layer"]

            image_count = page["image_object_count"]

            if has_text and image_count == 0:

                native_pages.append(
                    page["page_number"]
                )

            elif (not has_text) and image_count > 0:

                scan_pages.append(
                    page["page_number"]
                )

            else:

                mixed_pages.append(
                    page["page_number"]
                )

        native_count = len(native_pages)

        scan_count = len(scan_pages)

        mixed_count = len(mixed_pages)

        native_ratio = round(
            native_count / total_pages * 100,
            2
        )

        scan_ratio = round(
            scan_count / total_pages * 100,
            2
        )

        mixed_ratio = round(
            mixed_count / total_pages * 100,
            2
        )

        # ==========================
        # Determine PDF Type
        # ==========================

        if scan_count == 0 and mixed_count == 0:

            pdf_type = "NATIVE"

        elif native_count == 0 and mixed_count == 0:

            pdf_type = "SCAN"

        else:

            pdf_type = "HYBRID"

        # ==========================
        # OCR Strategy
        # ==========================

        if pdf_type == "NATIVE":

            ocr_required = False

            ocr_mode = "NONE"

        elif pdf_type == "SCAN":

            ocr_required = True

            ocr_mode = "FULL_DOCUMENT"

        else:

            ocr_required = True

            ocr_mode = "PAGE_BASED"

        # ==========================
        # Layout Strategy
        # ==========================

        if pdf_type == "NATIVE":

            layout_strategy = "TEXT_FIRST"

        elif pdf_type == "SCAN":

            layout_strategy = "OCR_FIRST"

        else:

            layout_strategy = "HYBRID"

        # ==========================
        # Processing Strategy
        # ==========================

        if pdf_type == "NATIVE":

            processing_strategy = "DIRECT_LAYOUT"

        elif pdf_type == "SCAN":

            processing_strategy = "OCR_LAYOUT"

        else:

            processing_strategy = "MIXED"

        result = {

            "pdf_type": pdf_type,

            "page_count": total_pages,

            "native_pages": native_count,

            "scan_pages": scan_count,

            "mixed_pages": mixed_count,

            "native_ratio": native_ratio,

            "scan_ratio": scan_ratio,

            "mixed_ratio": mixed_ratio,

            "ocr_required": ocr_required,

            "ocr_mode": ocr_mode,

            "layout_strategy": layout_strategy,

            "processing_strategy": processing_strategy,

            "page_groups": {

                "native_pages": native_pages,

                "scan_pages": scan_pages,

                "mixed_pages": mixed_pages

            }

        }

        return result

#### 2.1.2.3 Run

In [32]:
analyzer = PDFTypeAnalyzer(
    structure_profile
)

pdf_type_profile = analyzer.analyze()

print(
    json.dumps(
        pdf_type_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "pdf_type": "NATIVE",
  "page_count": 284,
  "native_pages": 284,
  "scan_pages": 0,
  "mixed_pages": 0,
  "native_ratio": 100.0,
  "scan_ratio": 0.0,
  "mixed_ratio": 0.0,
  "ocr_required": false,
  "ocr_mode": "NONE",
  "layout_strategy": "TEXT_FIRST",
  "processing_strategy": "DIRECT_LAYOUT",
  "page_groups": {
    "native_pages": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      6

#### 2.1.2.4 Save JSON

In [33]:
output_pdf_type_profile_file = RENDER_DIR /"2.pdf_type_profile.json"

with open(output_pdf_type_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pdf_type_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved")

Saved


### 2.1.3. Phân tích nội dung tài liệu

#### 2.1.3.1. Code

In [34]:
import fitz
import json

class DocumentContentAnalyzer:

    def __init__(
        self,
        pdf_file,
        structure_profile,
        pdf_type_profile
    ):

        self.pdf_file = pdf_file

        self.structure_profile = structure_profile

        self.pdf_type_profile = pdf_type_profile

        self.doc = fitz.open(pdf_file)

    def analyze(self):

        contains_text = False
        contains_image = False
        contains_table = False
        contains_form = False
        contains_hyperlink = False
        contains_bookmark = False
        contains_signature = False
        contains_stamp = False

        pages_with_text = 0
        pages_with_images = 0
        estimated_table_pages = 0
        pages_with_links = 0
        pages_with_forms = 0

        # =====================
        # Bookmark
        # =====================

        try:

            toc = self.doc.get_toc()

            if len(toc) > 0:

                contains_bookmark = True

        except Exception:

            pass

        # =====================
        # Page Analysis
        # =====================

        for page in self.doc:

            # -------------------
            # Text
            # -------------------

            text = page.get_text()

            if text.strip():

                contains_text = True

                pages_with_text += 1

            # -------------------
            # Images
            # -------------------

            images = page.get_images(
                full=True
            )

            if len(images) > 0:

                contains_image = True

                pages_with_images += 1

            # -------------------
            # Hyperlinks
            # -------------------

            links = page.get_links()

            if len(links) > 0:

                contains_hyperlink = True

                pages_with_links += 1

            # -------------------
            # Forms
            # -------------------

            try:

                widgets = page.widgets()

                if widgets:

                    contains_form = True

                    pages_with_forms += 1

            except Exception:

                pass

            # -------------------
            # Table Estimation
            #
            # Rule-based
            # -------------------

            text_dict = page.get_text(
                "dict"
            )

            block_count = len(
                text_dict.get(
                    "blocks",
                    []
                )
            )

            if block_count >= 20:

                estimated_table_pages += 1

                contains_table = True

        result = {

            "document_content_profile": {

                "contains_text":
                    contains_text,

                "contains_image":
                    contains_image,

                "contains_table":
                    contains_table,

                "contains_form":
                    contains_form,

                "contains_hyperlink":
                    contains_hyperlink,

                "contains_bookmark":
                    contains_bookmark,

                "contains_signature":
                    contains_signature,

                "contains_stamp":
                    contains_stamp
            },

            "content_statistics": {

                "pages_with_text":
                    pages_with_text,

                "pages_with_images":
                    pages_with_images,

                "estimated_table_pages":
                    estimated_table_pages,

                "pages_with_links":
                    pages_with_links,

                "pages_with_forms":
                    pages_with_forms
            }
        }

        return result

#### 2.1.3.2. Run

In [35]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

analyzer = DocumentContentAnalyzer(
    pdf_file,
    structure_profile,
    pdf_type_profile
)

content_profile = analyzer.analyze()

print(
    json.dumps(
        content_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_content_profile": {
    "contains_text": true,
    "contains_image": false,
    "contains_table": true,
    "contains_form": true,
    "contains_hyperlink": false,
    "contains_bookmark": false,
    "contains_signature": false,
    "contains_stamp": false
  },
  "content_statistics": {
    "pages_with_text": 284,
    "pages_with_images": 0,
    "estimated_table_pages": 184,
    "pages_with_links": 0,
    "pages_with_forms": 284
  }
}


#### 2.1.3.3. Lưu file

In [36]:
output_document_content_profile_file = RENDER_DIR /"3.document_content_profile.json"
with open(
    output_document_content_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        content_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

### 2.1.4. Phân tích phân bố dữ liệu

#### 2.1.4.1 Code

In [37]:
import json

class ContentDistributionAnalyzer:

    def __init__(
        self,
        structure_profile,
        pdf_type_profile,
        content_profile
    ):

        self.structure_profile = structure_profile

        self.pdf_type_profile = pdf_type_profile

        self.content_profile = content_profile

    def calculate_page_complexity(
        self,
        text_objects,
        image_objects,
        font_count,
        annotation_count
    ):

        score = (

            text_objects * 0.25

            +

            image_objects * 10

            +

            font_count * 2

            +

            annotation_count * 5

        )

        return min(
            round(score),
            100
        )

    def classify_complexity(
        self,
        score
    ):

        if score >= 80:

            return "HIGH"

        elif score >= 50:

            return "MEDIUM"

        else:

            return "LOW"

    def analyze(self):

        pages = self.structure_profile[
            "page_structure"
        ]

        page_profiles = []

        total_text_density = 0

        total_image_density = 0

        total_content_density = 0

        total_complexity = 0

        for page in pages:

            page_number = page[
                "page_number"
            ]

            text_objects = page.get(
                "text_object_count",
                0
            )

            image_objects = page.get(
                "image_object_count",
                0
            )

            font_count = page.get(
                "font_count",
                0
            )

            annotation_count = page.get(
                "annotation_count",
                0
            )

            total_objects = max(
                text_objects + image_objects,
                1
            )

            text_density = round(
                text_objects
                / total_objects
                * 100,
                2
            )

            image_density = round(
                image_objects
                / total_objects
                * 100,
                2
            )

            content_density = min(
                round(
                    (
                        text_objects * 0.2
                        +
                        image_objects * 8
                    )
                ),
                100
            )

            complexity_score = (
                self.calculate_page_complexity(
                    text_objects,
                    image_objects,
                    font_count,
                    annotation_count
                )
            )

            page_complexity = (
                self.classify_complexity(
                    complexity_score
                )
            )

            page_profiles.append({

                "page_number":
                    page_number,

                "text_density":
                    text_density,

                "image_density":
                    image_density,

                "content_density":
                    content_density,

                "complexity_score":
                    complexity_score,

                "page_complexity":
                    page_complexity
            })

            total_text_density += text_density

            total_image_density += image_density

            total_content_density += content_density

            total_complexity += complexity_score

        page_count = len(
            page_profiles
        )

        avg_text_density = round(
            total_text_density
            / page_count,
            2
        )

        avg_image_density = round(
            total_image_density
            / page_count,
            2
        )

        avg_content_density = round(
            total_content_density
            / page_count,
            2
        )

        avg_complexity = round(
            total_complexity
            / page_count,
            2
        )

        if avg_complexity >= 80:

            document_complexity = "HIGH"

        elif avg_complexity >= 50:

            document_complexity = "MEDIUM"

        else:

            document_complexity = "LOW"

        result = {

            "document_distribution_profile": {

                "average_text_density":
                    avg_text_density,

                "average_image_density":
                    avg_image_density,

                "average_content_density":
                    avg_content_density,

                "average_complexity_score":
                    avg_complexity,

                "document_complexity":
                    document_complexity
            },

            "page_distribution_profiles":
                page_profiles
        }

        return result

#### 2.1.4.2 Run

In [38]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

with open(
    output_document_content_profile_file,
    "r",
    encoding="utf-8"
) as f:

    content_profile = json.load(f)

analyzer = ContentDistributionAnalyzer(
    structure_profile,
    pdf_type_profile,
    content_profile
)

distribution_profile = analyzer.analyze()

print(
    json.dumps(
        distribution_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_distribution_profile": {
    "average_text_density": 100.0,
    "average_image_density": 0.0,
    "average_content_density": 4.6,
    "average_complexity_score": 11.05,
    "document_complexity": "LOW"
  },
  "page_distribution_profiles": [
    {
      "page_number": 1,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 3,
      "complexity_score": 9,
      "page_complexity": "LOW"
    },
    {
      "page_number": 2,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 0,
      "complexity_score": 2,
      "page_complexity": "LOW"
    },
    {
      "page_number": 3,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 6,
      "complexity_score": 12,
      "page_complexity": "LOW"
    },
    {
      "page_number": 4,
      "text_density": 100.0,
      "image_density": 0.0,
      "content_density": 6,
      "complexity_score": 11,
      "page_complexity": "LOW"
    },
    {
      "pa

#### 2.1.4.3. Lưu kết quả

In [39]:
output_content_distribution_profile_file = RENDER_DIR /"4.content_distribution_profile.json"
with open(
    output_content_distribution_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        distribution_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved")

Saved


### 2.1.5. Sinh hồ sơ cấu trúc tài liệu

#### 2.1.5.1. Code

In [40]:
import json

class DocumentProfileBuilder:

    def __init__(
        self,
        structure_profile,
        pdf_type_profile,
        content_profile,
        distribution_profile
    ):

        self.structure_profile = structure_profile
        self.pdf_type_profile = pdf_type_profile
        self.content_profile = content_profile
        self.distribution_profile = distribution_profile

    def build(self):

        file_info = self.structure_profile.get(
            "file_info",
            {}
        )

        page_summary = self.structure_profile.get(
            "page_summary",
            {}
        )

        object_statistics = self.structure_profile.get(
            "object_statistics",
            {}
        )

        page_structure = self.structure_profile.get(
            "page_structure",
            []
        )

        document_content = self.content_profile.get(
            "document_content_profile",
            {}
        )

        distribution = self.distribution_profile.get(
            "document_distribution_profile",
            {}
        )

        document_profile = {

            "document_id":

                self.structure_profile.get(
                    "document_id",
                    "DOC001"
                ),

            # ==========================
            # File Information
            # ==========================

            "file_info": {

                "file_name":
                    file_info.get(
                        "file_name"
                    ),

                "page_count":
                    file_info.get(
                        "page_count"
                    ),

                "pdf_version":
                    file_info.get(
                        "pdf_version"
                    ),

                "pdf_type":
                    self.pdf_type_profile.get(
                        "pdf_type"
                    )
            },

            # ==========================
            # Processing Profile
            # ==========================

            "processing_profile": {

                "ocr_required":
                    self.pdf_type_profile.get(
                        "ocr_required"
                    ),

                "ocr_mode":
                    self.pdf_type_profile.get(
                        "ocr_mode"
                    ),

                "layout_strategy":
                    self.pdf_type_profile.get(
                        "layout_strategy"
                    ),

                "processing_strategy":
                    self.pdf_type_profile.get(
                        "processing_strategy"
                    )
            },

            # ==========================
            # Physical Profile
            # ==========================

            "physical_profile": {

                "contains_text":
                    document_content.get(
                        "contains_text"
                    ),

                "contains_image":
                    document_content.get(
                        "contains_image"
                    ),

                "contains_table":
                    document_content.get(
                        "contains_table"
                    ),

                "contains_form":
                    document_content.get(
                        "contains_form"
                    ),

                "contains_hyperlink":
                    document_content.get(
                        "contains_hyperlink"
                    ),

                "contains_bookmark":
                    document_content.get(
                        "contains_bookmark"
                    ),

                "contains_signature":
                    document_content.get(
                        "contains_signature"
                    ),

                "contains_stamp":
                    document_content.get(
                        "contains_stamp"
                    ),

                "pages_with_text":
                    page_summary.get(
                        "pages_with_text_layer"
                    ),

                "pages_with_images":
                    page_summary.get(
                        "pages_with_images"
                    ),

                "pages_with_annotations":
                    page_summary.get(
                        "pages_with_annotations"
                    ),

                "rotated_pages":
                    page_summary.get(
                        "rotated_pages"
                    )
            },

            # ==========================
            # Object Statistics
            # ==========================

            "object_statistics": {

                "total_text_objects":
                    object_statistics.get(
                        "total_text_objects"
                    ),

                "total_image_objects":
                    object_statistics.get(
                        "total_image_objects"
                    ),

                "total_font_objects":
                    object_statistics.get(
                        "total_font_objects"
                    ),

                "total_annotation_objects":
                    object_statistics.get(
                        "total_annotation_objects"
                    )
            },

            # ==========================
            # Distribution Profile
            # ==========================

            "distribution_profile": {

                "average_text_density":
                    distribution.get(
                        "average_text_density"
                    ),

                "average_image_density":
                    distribution.get(
                        "average_image_density"
                    ),

                "average_content_density":
                    distribution.get(
                        "average_content_density"
                    ),

                "average_complexity_score":
                    distribution.get(
                        "average_complexity_score"
                    ),

                "document_complexity":
                    distribution.get(
                        "document_complexity"
                    )
            },

            # ==========================
            # Page Structure
            # ==========================

            "page_structure":

                page_structure
        }

        return document_profile

#### 2.1.5.2. Run

In [41]:
with open(
    output_pdf_structure_profile_file,
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    output_pdf_type_profile_file,
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

with open(
    output_document_content_profile_file,
    "r",
    encoding="utf-8"
) as f:

    content_profile = json.load(f)

with open(
    output_content_distribution_profile_file,
    "r",
    encoding="utf-8"
) as f:

    distribution_profile = json.load(f)

builder = DocumentProfileBuilder(
    structure_profile,
    pdf_type_profile,
    content_profile,
    distribution_profile
)

document_profile = builder.build()

#### 2.1.5.3. Lưu kết quả

In [42]:
output_document_profile_file = RENDER_DIR /"5.document_profile.json"
with open(
    output_document_profile_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        document_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

## 2.2. Layout Pre-Detection
> **Mục tiêu**
> *   *Hiểu bố cục vật lý của từng trang tài liệu*
> *   *Xác định vị trí và ranh giới của các vùng nội dung*
> *   *Phân loại các đối tượng hiển thị trước khi OCR*
> *   *Xây dựng Layout Map và Layout Tree làm nền tảng cho các bước xử lý tiếp theo*

> **Bài toán cần giải quyết**: Một trang tài liệu có nhiều loại đối tượng khác nhau (*Heading, Paragraph, Table, Image, Header, Footer, Footnote, Signature, Stamp*)

> **Do đó**
> *   *OCR toàn trang sẽ làm mất cấu trúc*
> *   *Không xác định được ranh giới giữa các vùng*
> *   *Không thể xác định thứ tự đọc chính xác*
> *   *Khó nhận diện bảng nhiều trang*
  

### 2.2.1. Layout Strategy Selection

In [44]:
import json

class LayoutStrategySelector:

    def __init__(self, profile_path):

        with open(profile_path, "r", encoding="utf-8") as f:
            self.profile = json.load(f)

    def get_strategy(self):

        profile = self.profile["processing_profile"]

        return profile["layout_strategy"]

#### 2.2.2. Physical Object Extraction

###### 2.2.2.1. text_extractor

> Extract physical TEXT objects from a PyMuPDF page.

    Output schema intentionally contains only physical information.
    No logical classification (Heading, Paragraph...) is performed here



In [51]:
from __future__ import annotations

import logging
from typing import Any, Dict, List


logger = logging.getLogger(__name__)


class TextExtractor:
    """
    Extract physical TEXT objects from a PyMuPDF page.

    Output schema intentionally contains only physical information.
    No logical classification (Heading, Paragraph...) is performed here.
    """

    def __init__(self, page):
        self.page = page

    def extract(self) -> List[Dict[str, Any]]:
        """Main entry."""
        try:
            page_dict = self.page.get_text("dict")
        except Exception as ex:
            logger.exception("Unable to extract page text: %s", ex)
            return []

        objects: List[Dict[str, Any]] = []

        for block_index, block in enumerate(page_dict.get("blocks", [])):

            # Only text blocks
            if block.get("type", 0) != 0:
                continue

            bbox = block.get("bbox", (0, 0, 0, 0))

            text_object = {
                "physical_type": "TEXT",
                "block_index": block_index,
                "bbox": {
                    "x": round(bbox[0], 2),
                    "y": round(bbox[1], 2),
                    "width": round(bbox[2] - bbox[0], 2),
                    "height": round(bbox[3] - bbox[1], 2),
                },
                "lines": [],
                "attributes": {
                    "line_count": 0,
                    "span_count": 0,
                    "text_length": 0,
                },
            }

            full_text = []

            for line_index, line in enumerate(block.get("lines", [])):

                line_bbox = line.get("bbox", (0, 0, 0, 0))

                line_item = {
                    "line_index": line_index,
                    "bbox": {
                        "x": round(line_bbox[0], 2),
                        "y": round(line_bbox[1], 2),
                        "width": round(line_bbox[2] - line_bbox[0], 2),
                        "height": round(line_bbox[3] - line_bbox[1], 2),
                    },
                    "spans": [],
                }

                for span_index, span in enumerate(line.get("spans", [])):

                    txt = span.get("text", "")

                    span_bbox = span.get("bbox", (0, 0, 0, 0))

                    span_item = {
                        "span_index": span_index,
                        "text": txt,
                        "font": span.get("font", ""),
                        "font_size": span.get("size", 0),
                        "flags": span.get("flags", 0),
                        "color": span.get("color", 0),
                        "bbox": {
                            "x": round(span_bbox[0], 2),
                            "y": round(span_bbox[1], 2),
                            "width": round(span_bbox[2] - span_bbox[0], 2),
                            "height": round(span_bbox[3] - span_bbox[1], 2),
                        },
                    }

                    line_item["spans"].append(span_item)

                    text_object["attributes"]["span_count"] += 1
                    full_text.append(txt)

                text_object["lines"].append(line_item)

            text_object["attributes"]["line_count"] = len(text_object["lines"])
            text_object["attributes"]["text"] = "".join(full_text)
            text_object["attributes"]["text_length"] = len(
                text_object["attributes"]["text"]
            )

            objects.append(text_object)

        logger.info("Extracted %d text blocks", len(objects))
        return objects


##### 2.2.2.2. image_extractor

> Extract physical IMAGE objects from a PyMuPDF page.

    This extractor only collects image metadata and bounding boxes.
    It does not export image bytes by default.



In [52]:
from __future__ import annotations

import hashlib
import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class ImageExtractor:
    """
    Extract physical IMAGE objects from a PyMuPDF page.

    This extractor only collects image metadata and bounding boxes.
    It does not export image bytes by default.
    """

    def __init__(self, page):
        self.page = page

    def _image_blocks(self) -> List[Dict[str, Any]]:
        """Extract image blocks from page.get_text('dict')."""
        page_dict = self.page.get_text("dict")
        objects = []

        for block_index, block in enumerate(page_dict.get("blocks", [])):
            if block.get("type") != 1:
                continue

            bbox = block.get("bbox", (0, 0, 0, 0))

            obj = {
                "physical_type": "IMAGE",
                "block_index": block_index,
                "bbox": {
                    "x": round(bbox[0], 2),
                    "y": round(bbox[1], 2),
                    "width": round(bbox[2] - bbox[0], 2),
                    "height": round(bbox[3] - bbox[1], 2),
                },
                "attributes": {
                    "width_px": block.get("width"),
                    "height_px": block.get("height"),
                    "colorspace": block.get("colorspace"),
                    "xres": block.get("xres"),
                    "yres": block.get("yres"),
                    "bpc": block.get("bpc"),
                    "transform": block.get("transform"),
                    "size_bytes": len(block.get("image", b"")) if block.get("image") else 0,
                },
            }

            if block.get("image"):
                obj["attributes"]["image_hash"] = hashlib.md5(
                    block["image"]
                ).hexdigest()

            objects.append(obj)

        return objects

    def _image_resources(self) -> List[Dict[str, Any]]:
        """
        Collect image resource information from page.get_images().
        Used for validation and future image export.
        """
        resources = []

        try:
            for item in self.page.get_images(full=True):
                resources.append(
                    {
                        "xref": item[0],
                        "width": item[2],
                        "height": item[3],
                        "bpc": item[4],
                        "colorspace": item[5],
                        "filter": item[8],
                    }
                )
        except Exception as ex:
            logger.warning("Unable to enumerate page images: %s", ex)

        return resources

    def extract(self) -> List[Dict[str, Any]]:
        """
        Return IMAGE physical objects.
        """

        objects = self._image_blocks()

        resources = self._image_resources()

        logger.info(
            "Extracted %d image blocks (%d image resources)",
            len(objects),
            len(resources),
        )

        # Resources retained for future enhancement if needed.
        return objects


##### 2.2.2.3. table_extractor

> Extract TABLE regions from a PDF page using PyMuPDF.

This extractor only detects physical table regions.
It does NOT recognize cells or classify logical structure.



In [53]:
from __future__ import annotations

import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class TableExtractor:

    def __init__(self, page):
        self.page = page

    def _build_table_object(self, table, table_index: int) -> Dict[str, Any]:
        x0, y0, x1, y1 = table.bbox

        obj = {
            "physical_type": "TABLE",
            "table_index": table_index,
            "bbox": {
                "x": round(x0, 2),
                "y": round(y0, 2),
                "width": round(x1 - x0, 2),
                "height": round(y1 - y0, 2),
            },
            "attributes": {
                "row_count": getattr(table, "row_count", None),
                "column_count": getattr(table, "col_count", None),
                "cell_count": None,
                "has_header": False,
                "confidence": 1.0,
            },
            "children": [],
            "parent_object": None,
            "logical_type": None,
        }

        rows = getattr(table, "row_count", None)
        cols = getattr(table, "col_count", None)
        if rows is not None and cols is not None:
            obj["attributes"]["cell_count"] = rows * cols

        try:
            header = getattr(table, "header", None)
            if header:
                obj["attributes"]["has_header"] = True
        except Exception:
            pass

        return obj

    def extract(self) -> List[Dict[str, Any]]:
        objects: List[Dict[str, Any]] = []

        try:
            result = self.page.find_tables()
        except Exception as ex:
            logger.warning("find_tables() failed: %s", ex)
            return objects

        tables = getattr(result, "tables", [])

        for idx, table in enumerate(tables):
            try:
                objects.append(self._build_table_object(table, idx))
            except Exception as ex:
                logger.warning("Skip table %d: %s", idx, ex)

        logger.info("Detected %d table(s)", len(objects))
        return objects


##### 2.2.2.4. drawing_extractor

> Extract vector drawing objects from a PDF page using PyMuPDF.

Purpose
-------
Collect physical vector graphics that may later help detect:
- Table borders
- Horizontal/vertical separators
- Underlines
- Shapes
- Graphic decorations

This module performs NO logical classification.



In [54]:
from __future__ import annotations

import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class DrawingExtractor:
    """
    Extract vector drawing information from a PDF page.
    """

    def __init__(self, page):
        self.page = page

    def _detect_shape_type(self, drawing: Dict[str, Any]) -> str:
        """
        Roughly infer drawing type from path items.
        """
        items = drawing.get("items", [])

        if not items:
            return "UNKNOWN"

        commands = [str(i[0]).lower() for i in items]

        if all(c == "l" for c in commands):
            return "LINE"

        if any(c == "re" for c in commands):
            return "RECTANGLE"

        if any(c == "c" for c in commands):
            return "CURVE"

        return "PATH"

    def extract(self) -> List[Dict[str, Any]]:

        objects: List[Dict[str, Any]] = []

        try:
            drawings = self.page.get_drawings()
        except Exception as ex:
            logger.warning("Unable to extract drawings: %s", ex)
            return objects

        for idx, drawing in enumerate(drawings):

            rect = drawing.get("rect")

            if rect is None:
                continue

            obj = {
                "physical_type": "DRAWING",
                "drawing_index": idx,
                "bbox": {
                    "x": round(rect.x0, 2),
                    "y": round(rect.y0, 2),
                    "width": round(rect.x1 - rect.x0, 2),
                    "height": round(rect.y1 - rect.y0, 2),
                },
                "attributes": {
                    "shape_type": self._detect_shape_type(drawing),
                    "stroke_color": drawing.get("color"),
                    "fill_color": drawing.get("fill"),
                    "line_width": drawing.get("width"),
                    "line_cap": drawing.get("lineCap"),
                    "line_join": drawing.get("lineJoin"),
                    "dashes": drawing.get("dashes"),
                    "fill_opacity": drawing.get("fill_opacity"),
                    "stroke_opacity": drawing.get("stroke_opacity"),
                    "item_count": len(drawing.get("items", [])),
                    "close_path": drawing.get("closePath"),
                    "even_odd": drawing.get("even_odd"),
                },
                "children": [],
                "parent_object": None,
                "logical_type": None,
            }

            objects.append(obj)

        logger.info("Extracted %d drawing object(s)", len(objects))

        return objects


##### 2.2.2.5. link_extractor

> Extract hyperlink objects from a PDF page using PyMuPDF.

Purpose
-------
Collect physical hyperlink regions for later layout and semantic analysis.

This module does NOT classify logical structure.



In [55]:
from __future__ import annotations

import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class LinkExtractor:
    """
    Extract hyperlink annotations from a PDF page.
    """

    def __init__(self, page):
        self.page = page

    def extract(self) -> List[Dict[str, Any]]:

        objects: List[Dict[str, Any]] = []

        try:
            links = self.page.get_links()
        except Exception as ex:
            logger.warning("Unable to extract links: %s", ex)
            return objects

        for idx, link in enumerate(links):

            rect = link.get("from")

            if rect is None:
                continue

            obj = {
                "physical_type": "LINK",
                "link_index": idx,
                "bbox": {
                    "x": round(rect.x0, 2),
                    "y": round(rect.y0, 2),
                    "width": round(rect.x1 - rect.x0, 2),
                    "height": round(rect.y1 - rect.y0, 2),
                },
                "attributes": {
                    "kind": link.get("kind"),
                    "uri": link.get("uri"),
                    "page": link.get("page"),
                    "to": str(link.get("to")) if link.get("to") is not None else None,
                    "xref": link.get("xref"),
                    "id": link.get("id"),
                    "is_external": bool(link.get("uri")),
                },
                "children": [],
                "parent_object": None,
                "logical_type": None,
            }

            objects.append(obj)

        logger.info("Extracted %d link object(s)", len(objects))
        return objects


##### 2.2.2.6. annotation_extractor

> Extract PDF annotation objects using PyMuPDF.

Purpose
-------
Collect physical annotation regions for downstream layout analysis.
No logical classification is performed here.

Supported examples:
- Highlight
- Underline
- Strikeout
- Squiggly
- Text Note
- FreeText
- Ink
- Stamp
- Square / Circle



In [56]:
from __future__ import annotations

import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class AnnotationExtractor:
    """
    Extract annotation objects from a PDF page.
    """

    def __init__(self, page):
        self.page = page

    @staticmethod
    def _annotation_type(annot) -> str:
        try:
            info = annot.type
            if isinstance(info, tuple):
                return str(info[1])
            return str(info)
        except Exception:
            return "UNKNOWN"

    def extract(self) -> List[Dict[str, Any]]:

        objects: List[Dict[str, Any]] = []

        try:
            annots = self.page.annots()
        except Exception as ex:
            logger.warning("Unable to enumerate annotations: %s", ex)
            return objects

        if annots is None:
            return objects

        for idx, annot in enumerate(annots):

            try:
                rect = annot.rect
            except Exception:
                continue

            info = {}
            try:
                info = annot.info or {}
            except Exception:
                pass

            obj = {
                "physical_type": "ANNOTATION",
                "annotation_index": idx,
                "bbox": {
                    "x": round(rect.x0, 2),
                    "y": round(rect.y0, 2),
                    "width": round(rect.x1 - rect.x0, 2),
                    "height": round(rect.y1 - rect.y0, 2),
                },
                "attributes": {
                    "annotation_type": self._annotation_type(annot),
                    "title": info.get("title"),
                    "subject": info.get("subject"),
                    "content": info.get("content"),
                    "name": info.get("name"),
                    "creation_date": info.get("creationDate"),
                    "modification_date": info.get("modDate"),
                    "flags": getattr(annot, "flags", None),
                    "opacity": getattr(annot, "opacity", None),
                    "colors": getattr(annot, "colors", None),
                    "border": getattr(annot, "border", None),
                },
                "children": [],
                "parent_object": None,
                "logical_type": None,
            }

            objects.append(obj)

        logger.info("Extracted %d annotation(s)", len(objects))
        return objects


##### 2.2.2.7. object_extractor

> Aggregate all physical object extractors into a unified object list.



In [45]:
from __future__ import annotations

import logging
from typing import Any, Dict, List

logger = logging.getLogger(__name__)


class ObjectExtractor:
    """
    Collect all physical objects detected on a PDF page.

    Output:
        List[Object]
    """

    def __init__(self, page):
        self.page = page

    def extract_text(self) -> List[Dict[str, Any]]:
        return TextExtractor(self.page).extract()

    def extract_images(self) -> List[Dict[str, Any]]:
        return ImageExtractor(self.page).extract()

    def extract_tables(self) -> List[Dict[str, Any]]:
        return TableExtractor(self.page).extract()

    def extract_drawings(self) -> List[Dict[str, Any]]:
        return DrawingExtractor(self.page).extract()

    def extract_links(self) -> List[Dict[str, Any]]:
        return LinkExtractor(self.page).extract()

    def extract_annotations(self) -> List[Dict[str, Any]]:
        return AnnotationExtractor(self.page).extract()

    @staticmethod
    def spatial_sort(objects: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return sorted(
            objects,
            key=lambda o: (
                o["bbox"]["y"],
                o["bbox"]["x"]
            )
        )

    @staticmethod
    def build_summary(objects: List[Dict[str, Any]]) -> Dict[str, Any]:
        summary = {
            "TEXT": 0,
            "IMAGE": 0,
            "TABLE": 0,
            "DRAWING": 0,
            "LINK": 0,
            "ANNOTATION": 0,
            "UNKNOWN": 0,
        }

        for obj in objects:
            t = obj.get("physical_type", "UNKNOWN")
            summary[t] = summary.get(t, 0) + 1

        summary["TOTAL"] = len(objects)
        return summary

    def extract(self) -> Dict[str, Any]:
        objects: List[Dict[str, Any]] = []

        logger.info("Extracting TEXT...")
        objects.extend(self.extract_text())

        logger.info("Extracting IMAGE...")
        objects.extend(self.extract_images())

        logger.info("Extracting TABLE...")
        objects.extend(self.extract_tables())

        logger.info("Extracting DRAWING...")
        objects.extend(self.extract_drawings())

        logger.info("Extracting LINK...")
        objects.extend(self.extract_links())

        logger.info("Extracting ANNOTATION...")
        objects.extend(self.extract_annotations())

        objects = self.spatial_sort(objects)

        summary = self.build_summary(objects)

        logger.info("Extraction summary: %s", summary)

        return {
            "summary": summary,
            "objects": objects,
        }


##### 2.2.2.8. object_processor

> Normalize and enrich physical objects after extraction.

Responsibilities
----------------
1. Assign unique object_id
2. Spatial sort
3. Associate objects (TEXT inside TABLE)
4. Assign region_id
5. Build page metadata



In [80]:
from __future__ import annotations

from typing import Any, Dict, List


class ObjectProcessor:

    def __init__(self, page_number: int):
        self.page_number = page_number

    @staticmethod
    def _inside(inner: Dict[str, float], outer: Dict[str, float], tol: float = 2.0) -> bool:
        return (
            inner["x"] >= outer["x"] - tol
            and inner["y"] >= outer["y"] - tol
            and inner["x"] + inner["width"] <= outer["x"] + outer["width"] + tol
            and inner["y"] + inner["height"] <= outer["y"] + outer["height"] + tol
        )

    def assign_object_ids(self, objects: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        for idx, obj in enumerate(objects, start=1):
            obj["object_id"] = f"OBJ{self.page_number:04d}{idx:05d}"
            obj["page_number"] = self.page_number
            obj.setdefault("logical_type", None)
            obj.setdefault("parent_object", None)
            obj.setdefault("children", [])
            obj.setdefault("region_id", None)
        return objects

    @staticmethod
    def spatial_sort(objects: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return sorted(
            objects,
            key=lambda o: (
                round(o["bbox"]["y"], 1),
                round(o["bbox"]["x"], 1)
            ),
        )

    def associate_tables(self, objects: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        tables = [o for o in objects if o["physical_type"] == "TABLE"]

        table_map = {t["object_id"]: t for t in tables}

        for t in tables:
            t["children"] = []

        for obj in objects:

            if obj["physical_type"] == "TABLE":
                obj["region_id"] = obj["object_id"]
                continue

            for table in tables:

                if self._inside(obj["bbox"], table["bbox"]):
                    obj["parent_object"] = table["object_id"]
                    obj["region_id"] = table["object_id"]
                    table_map[table["object_id"]]["children"].append(obj["object_id"])
                    break

            if obj["region_id"] is None:
                obj["region_id"] = obj["object_id"]

        return objects

    def build_metadata(self, objects: List[Dict[str, Any]]) -> Dict[str, Any]: # Corrected line: added self
        metadata = {
            "page_number": self.page_number,
            "object_count": len(objects),
            "text_count": 0,
            "image_count": 0,
            "table_count": 0,
            "drawing_count": 0,
            "link_count": 0,
            "annotation_count": 0
        }

        mapping = {
            "TEXT": "text_count",
            "IMAGE": "image_count",
            "TABLE": "table_count",
            "DRAWING": "drawing_count",
            "LINK": "link_count",
            "ANNOTATION": "annotation_count",
        }

        for obj in objects:
            key = mapping.get(obj["physical_type"])
            if key:
                metadata[key] += 1

        return metadata

    def process(self, extracted: Dict[str, Any]) -> Dict[str, Any]:
        objects = extracted["objects"]

        objects = self.assign_object_ids(objects)
        objects = self.spatial_sort(objects)
        objects = self.associate_tables(objects)

        return {
            "metadata": self.build_metadata(objects),
            "objects": objects,
        }

### 2.2.3. Region builder

> Build physical regions from processed objects.

This module groups physical objects into higher-level layout regions.
It does NOT perform semantic understanding.

Input:
    {
        "metadata": {...},
        "objects": [...]
    }

Output:
    {
        "regions": [...],
        "metadata": {...}
    }



In [83]:
from __future__ import annotations

from typing import Any, Dict, List


class RegionBuilder:

    def __init__(self, y_gap: float = 20.0):
        self.y_gap = y_gap

    @staticmethod
    def _merge_bbox(boxes: List[Dict[str, float]]) -> Dict[str, float]:
        x0 = min(b["x"] for b in boxes)
        y0 = min(b["y"] for b in boxes)
        x1 = max(b["x"] + b["width"] for b in boxes)
        y1 = max(b["y"] + b["height"] for b in boxes)

        return {
            "x": round(x0, 2),
            "y": round(y0, 2),
            "width": round(x1 - x0, 2),
            "height": round(y1 - y0, 2),
        }

    def _region_type(self, objs: List[Dict[str, Any]]) -> str:
        physical = {o["physical_type"] for o in objs}

        if "TABLE" in physical:
            return "TABLE_REGION"

        if "IMAGE" in physical:
            return "IMAGE_REGION"

        return "TEXT_REGION"

    def build(self, processed: Dict[str, Any]) -> Dict[str, Any]:

        objects = sorted(
            processed["objects"],
            key=lambda o: (
                o["bbox"]["y"],
                o["bbox"]["x"]
            )
        )

        regions: List[Dict[str, Any]] = []

        current = []

        region_id = 1

        last_bottom = None

        for obj in objects:
            # ------------------------------------
            # Skip object đã thuộc object khác
            # ------------------------------------
            if obj["parent_object"] is not None:
              continue
            # Table luôn là một Region độc lập
            if obj["physical_type"] == "TABLE":

                if current:
                    regions.append(self._create_region(current, region_id))
                    region_id += 1
                    current = []
                    last_bottom = None

                regions.append(self._create_region([obj], region_id))
                region_id += 1
                continue

            top = obj["bbox"]["y"]
            bottom = obj["bbox"]["y"] + obj["bbox"]["height"]

            if not current:
                current.append(obj)
                last_bottom = bottom
                continue

            if top - last_bottom <= self.y_gap:
                current.append(obj)
                last_bottom = max(last_bottom, bottom)
            else:
                regions.append(self._create_region(current, region_id))
                region_id += 1
                current = [obj]
                last_bottom = bottom

        if current:
            regions.append(self._create_region(current, region_id))

        return {
            "metadata": {
                **processed["metadata"],
                "region_count": len(regions)
            },
            "regions": regions
        }

    def _create_region(self, objs: List[Dict[str, Any]], region_id: int):

        bbox = self._merge_bbox([o["bbox"] for o in objs])

        rid = f"REGION{region_id:05d}"

        for o in objs:
            o["region_id"] = rid

        return {
            "region_id": rid,
            "region_type": self._region_type(objs),
            "bbox": bbox,
            "object_count": len(objs),
            "object_ids": [o["object_id"] for o in objs],
            "physical_types": list({o["physical_type"] for o in objs}),
            "logical_type": None,
            "confidence": 1.0
        }


### 2.2.4. layout tree builder

> Build a layout tree from detected regions.



In [84]:
from typing import Dict, Any


class LayoutTreeBuilder:

    def build(self, region_result: Dict[str, Any]) -> Dict[str, Any]:

        regions = sorted(
            region_result["regions"],
            key=lambda r: (
                r["bbox"]["y"],
                r["bbox"]["x"]
            )
        )

        tree = {
            "node_type": "PAGE",
            "children": []
        }

        reading_order = 1

        for region in regions:

            node = {

                "node_id": region["region_id"],

                "node_type": region["region_type"],

                "reading_hint": reading_order,

                "bbox": region["bbox"],

                "logical_type": region.get("logical_type"),

                "confidence": region.get("confidence", 1.0),

                "object_count": region.get("object_count", 0),

                "physical_types": region.get("physical_types", []),

                "children": []

            }

            # Build Object Node
            for child in region.get("children", []):

                node["children"].append({

                    "node_id": child["object_id"],

                    "node_type": child["physical_type"],

                    "bbox": child["bbox"],

                    "logical_type": None

                })

            tree["children"].append(node)

            reading_order += 1

        return {

            "metadata": region_result["metadata"],

            "layout_tree": tree

        }

### 2.2.5. page layout profiler

> Generate page-level layout statistics from Layout Tree.



In [85]:
from typing import Dict, Any


class PageLayoutProfiler:

    def build(self, layout_result: Dict[str, Any]) -> Dict[str, Any]:

        tree = layout_result["layout_tree"]

        profile = {
            "region_count": 0,
            "text_region_count": 0,
            "table_region_count": 0,
            "image_region_count": 0,
            "header_candidate": False,
            "footer_candidate": False,
            "dominant_layout": "SINGLE_COLUMN"
        }

        children = tree["children"]
        profile["region_count"] = len(children)

        ys = []

        for node in children:
            t = node["node_type"]

            if t == "TEXT_REGION":
                profile["text_region_count"] += 1
            elif t == "TABLE_REGION":
                profile["table_region_count"] += 1
            elif t == "IMAGE_REGION":
                profile["image_region_count"] += 1

            ys.append(node["bbox"]["y"])

        if ys:
            if min(ys) < 60:
                profile["header_candidate"] = True
            if max(ys) > 700:
                profile["footer_candidate"] = True

        if profile["table_region_count"] > 3:
            profile["dominant_layout"] = "TABLE_HEAVY"

        return {
            "metadata": layout_result["metadata"],
            "layout_profile": profile,
            "layout_tree": tree
        }


### 2.2.6. Layout pipeline

In [86]:
import os
import json
import fitz

class LayoutPipeline:

    def __init__(self, pdf_path, output_dir):

        self.pdf_path = pdf_path
        self.output_dir = output_dir

        self.doc = fitz.open(pdf_path)

        os.makedirs(output_dir, exist_ok=True)

    # =====================================================
    # Process One Page
    # =====================================================

    def process_page(self, page_index):

        page = self.doc.load_page(page_index)

        print(f"Processing Page {page_index+1}")

        # -------------------------------------------------
        # Step 1
        # Object Extraction
        # -------------------------------------------------

        extracted = ObjectExtractor(page).extract()

        # -------------------------------------------------
        # Step 2
        # Object Processing
        # -------------------------------------------------

        processed = ObjectProcessor(
            page_number=page_index + 1
        ).process(extracted)

        # -------------------------------------------------
        # Step 3
        # Region Builder
        # -------------------------------------------------

        regions = RegionBuilder().build(processed)

        # -------------------------------------------------
        # Step 4
        # Layout Tree
        # -------------------------------------------------

        tree = LayoutTreeBuilder().build(regions)

        # -------------------------------------------------
        # Step 5
        # Page Layout Profile
        # -------------------------------------------------

        result = PageLayoutProfiler().build(tree)

        page_result = {
            "page_number": page_index + 1,
            "page_size":{
                "width": round(page.rect.width,2),
                "height": round(page.rect.height,2)
            },
            "metadata": result["metadata"],
            "layout_profile": result["layout_profile"],
            "layout_tree": result["layout_tree"]
        }

        return page_result

    # =====================================================
    # Save
    # =====================================================

    def save_page(self, page_result):

        filename = os.path.join(

            self.output_dir,

            f"page_{page_result['metadata']['page_number']:06d}.json"

        )

        with open(

            filename,

            "w",

            encoding="utf-8"

        ) as f:

            json.dump(

                page_result,

                f,

                ensure_ascii=False,

                indent=4

            )

        print(f"Saved : {filename}")

    # =====================================================
    # Run
    # =====================================================

    def run(self):

        print("=" * 80)
        print("Layout Pre-Detection Pipeline")
        print("=" * 80)

        print(f"PDF : {self.pdf_path}")
        print(f"Pages : {len(self.doc)}")

        for page_index in range(len(self.doc)):

            result = self.process_page(page_index)

            self.save_page(result)

        print("=" * 80)
        print("Finished")

### 2.2.7. Layout Strategy Selection

In [87]:
selector = LayoutStrategySelector(
    output_document_profile_file
)

strategy = selector.get_strategy()

if strategy == "TEXT_FIRST":

    print("Native PDF Layout")

    pipeline = LayoutPipeline(

        pdf_path="SachBCTHKQKT2021.pdf",

        output_dir="output/page_layout"

    )

    pipeline.run()

elif strategy == "OCR_FIRST":

    print("Vision AI Layout")

    #pipeline = VisionLayoutPipeline(

        #pdf_path="SachBCTHKQKT2021.pdf",

        #output_dir="output/page_layout"

    #)

    #pipeline.run()

elif strategy == "HYBRID":

    print("Hybrid Layout")

    #pipeline = HybridLayoutPipeline(

        #pdf_path="SachBCTHKQKT2021.pdf",

        #output_dir="output/page_layout"

    #)

    #pipeline.run()

else:

    raise Exception("Unknown Layout Strategy")

Native PDF Layout
Layout Pre-Detection Pipeline
PDF : SachBCTHKQKT2021.pdf
Pages : 284
Processing Page 1
Saved : output/page_layout/page_000001.json
Processing Page 2
Saved : output/page_layout/page_000002.json
Processing Page 3
Saved : output/page_layout/page_000003.json
Processing Page 4
Saved : output/page_layout/page_000004.json
Processing Page 5
Saved : output/page_layout/page_000005.json
Processing Page 6
Saved : output/page_layout/page_000006.json
Processing Page 7
Saved : output/page_layout/page_000007.json
Processing Page 8
Saved : output/page_layout/page_000008.json
Processing Page 9
Saved : output/page_layout/page_000009.json
Processing Page 10
Saved : output/page_layout/page_000010.json
Processing Page 11
Saved : output/page_layout/page_000011.json
Processing Page 12
Saved : output/page_layout/page_000012.json
Processing Page 13
Saved : output/page_layout/page_000013.json
Processing Page 14
Saved : output/page_layout/page_000014.json
Processing Page 15
Saved : output/page_l

### 2.2.8. Build Layout Document Manifest

> Input:
    output/page_layout/page_*.json

> Output:
    output/layout_document.json



In [100]:
import os
import json
import glob


class LayoutDocumentBuilder:

    def __init__(self,
                 layout_dir: str,
                 output_file: str):

        self.layout_dir = layout_dir
        self.output_file = output_file

    # =======================================================
    # Build
    # =======================================================

    def build(self):

        page_files = sorted(
            glob.glob(
                os.path.join(
                    self.layout_dir,
                    "page_*.json"
                )
            )
        )

        document = {

            "document_id": "DOC001",

            "page_count": len(page_files),

            "pages": [],

            "statistics": {

                "text_pages": 0,

                "table_pages": 0,

                "image_pages": 0,

                "drawing_pages": 0,

                "total_objects": 0,

                "total_regions": 0

            }

        }

        for file in page_files:

            with open(file, "r", encoding="utf-8") as f:

                page = json.load(f)

            metadata = page["metadata"]

            profile = page["layout_profile"]

            page_info = {

                "page_number": metadata["page_number"],

                "layout_file": os.path.basename(file),

                "object_count": metadata["object_count"],

                "region_count": metadata["region_count"],

                "has_table":
                    metadata["table_count"] > 0,

                "has_image":
                    metadata["image_count"] > 0,

                "has_drawing":
                    metadata["drawing_count"] > 0,

                "layout_type":
                    profile["dominant_layout"]

            }

            document["pages"].append(page_info)

            # -------------------------------
            # Statistics
            # -------------------------------

            document["statistics"]["total_objects"] += metadata["object_count"]

            document["statistics"]["total_regions"] += metadata["region_count"]

            if metadata["text_count"] > 0:
                document["statistics"]["text_pages"] += 1

            if metadata["table_count"] > 0:
                document["statistics"]["table_pages"] += 1

            if metadata["image_count"] > 0:
                document["statistics"]["image_pages"] += 1

            if metadata["drawing_count"] > 0:
                document["statistics"]["drawing_pages"] += 1

        return document

    # =======================================================
    # Save
    # =======================================================

    def save(self):

        document = self.build()

        os.makedirs(
            os.path.dirname(self.output_file),
            exist_ok=True
        )

        with open(
            self.output_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                document,
                f,
                ensure_ascii=False,
                indent=4
            )

        print(f"Saved: {self.output_file}")

        return document

## 2.2.9. Run merge

In [101]:
builder = LayoutDocumentBuilder(

    layout_dir="output/page_layout",

    output_file="output/layout_document.json"

)

builder.save()

Saved: output/layout_document.json


{'document_id': 'DOC001',
 'page_count': 284,
 'pages': [{'page_number': 1,
   'layout_file': 'page_000001.json',
   'object_count': 14,
   'region_count': 2,
   'has_table': False,
   'has_image': False,
   'has_drawing': True,
   'layout_type': 'SINGLE_COLUMN'},
  {'page_number': 2,
   'layout_file': 'page_000002.json',
   'object_count': 2,
   'region_count': 1,
   'has_table': False,
   'has_image': False,
   'has_drawing': True,
   'layout_type': 'SINGLE_COLUMN'},
  {'page_number': 3,
   'layout_file': 'page_000003.json',
   'object_count': 352,
   'region_count': 2,
   'has_table': True,
   'has_image': False,
   'has_drawing': True,
   'layout_type': 'SINGLE_COLUMN'},
  {'page_number': 4,
   'layout_file': 'page_000004.json',
   'object_count': 326,
   'region_count': 3,
   'has_table': True,
   'has_image': False,
   'has_drawing': True,
   'layout_type': 'SINGLE_COLUMN'},
  {'page_number': 5,
   'layout_file': 'page_000005.json',
   'object_count': 29,
   'region_count': 3,
  

### 2.2.1. Page Rendering (tạm thời)

> Kết xuất các trạng PDF theo document_profile.json từ bước 1



#### 2.2.1.1. Config

In [88]:
import json
import time
import hashlib
from pathlib import Path
import fitz
from PIL import Image
from pathlib import Path
OUTPUT_DIR = Path("output")

In [89]:
RENDER_DIR = OUTPUT_DIR / "1.Physical_Analysis" / "2.Layout_Pre_Detection"
RENDER_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DIR = RENDER_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

In [90]:
PDF_FILE = pdf_file
DOCUMENT_PROFILE = output_document_profile_file
PAGE_RENDER_PROFILE = RENDER_DIR / "1.page_render_profile.json"

#### 2.2.1.2. Process

In [91]:
class PageRenderer:

    def __init__(self, pdf_file, profile_file):
        self.pdf_file = Path(pdf_file)
        self.profile_file = Path(profile_file)

        if not self.pdf_file.exists():
            raise FileNotFoundError(self.pdf_file)

        if not self.profile_file.exists():
            raise FileNotFoundError(self.profile_file)

        with open(self.profile_file, "r", encoding="utf-8") as f:
            self.profile = json.load(f)

        self.doc = fitz.open(self.pdf_file)

    def build_strategy(self):
        processing = self.profile.get("processing_profile", {})
        file_info = self.profile.get("file_info", {})

        pdf_type = file_info.get("pdf_type", "NATIVE")

        if pdf_type == "SCAN":
            dpi = 300
            color = "L"
        elif pdf_type == "HYBRID":
            dpi = 250
            color = "RGB"
        else:
            dpi = 200
            color = "RGB"

        return {
            "pdf_type": pdf_type,
            "dpi": dpi,
            "color_mode": color,
            "ocr_mode": processing.get("ocr_mode", "NONE"),
            "processing_strategy": processing.get(
                "processing_strategy",
                "DIRECT_LAYOUT"
            )
        }

    @staticmethod
    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            while True:
                b = f.read(1024 * 1024)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()

    def render(self):
        strategy = self.build_strategy()

        scale = strategy["dpi"] / 72.0

        result = {
            "render_engine": "PyMuPDF",
            "render_strategy": strategy,
            "pages": []
        }

        total = len(self.doc)

        for idx in range(total):

            page = self.doc.load_page(idx)

            start = time.time()

            pix = page.get_pixmap(
                matrix=fitz.Matrix(scale, scale),
                alpha=False
            )

            image_path = IMAGE_DIR / f"page_{idx+1:05d}.png"

            pix.save(str(image_path))

            if strategy["color_mode"] == "L":
                img = Image.open(image_path).convert("L")
                img.save(image_path)

            elapsed = round((time.time() - start) * 1000, 2)

            result["pages"].append({
                "page_number": idx + 1,
                "image_file": image_path.name,
                "width": pix.width,
                "height": pix.height,
                "dpi": strategy["dpi"],
                "render_time_ms": elapsed,
                "sha256": self.sha256_file(image_path)
            })

            print(f"Rendered {idx+1}/{total}")

            del pix

        with open(PAGE_RENDER_PROFILE, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=4)

        print("Done.")
        print("Profile:", PAGE_RENDER_PROFILE)


#### 2.2.1.3. Run

In [92]:
renderer = PageRenderer(
    PDF_FILE,
    DOCUMENT_PROFILE
)
renderer.render()

Rendered 1/284
Rendered 2/284
Rendered 3/284
Rendered 4/284
Rendered 5/284
Rendered 6/284
Rendered 7/284
Rendered 8/284
Rendered 9/284
Rendered 10/284
Rendered 11/284
Rendered 12/284
Rendered 13/284
Rendered 14/284
Rendered 15/284
Rendered 16/284
Rendered 17/284
Rendered 18/284
Rendered 19/284
Rendered 20/284
Rendered 21/284
Rendered 22/284
Rendered 23/284
Rendered 24/284
Rendered 25/284
Rendered 26/284
Rendered 27/284
Rendered 28/284
Rendered 29/284
Rendered 30/284
Rendered 31/284
Rendered 32/284
Rendered 33/284
Rendered 34/284
Rendered 35/284
Rendered 36/284
Rendered 37/284
Rendered 38/284
Rendered 39/284
Rendered 40/284
Rendered 41/284
Rendered 42/284
Rendered 43/284
Rendered 44/284
Rendered 45/284
Rendered 46/284
Rendered 47/284
Rendered 48/284
Rendered 49/284
Rendered 50/284
Rendered 51/284
Rendered 52/284
Rendered 53/284
Rendered 54/284
Rendered 55/284
Rendered 56/284
Rendered 57/284
Rendered 58/284
Rendered 59/284
Rendered 60/284
Rendered 61/284
Rendered 62/284
Rendered 63/284
R

# 3. Structure Reconstruction

## 3.1. Reading Order Reconstruction

In [93]:
"""
reading_order_builder.py
"""

from __future__ import annotations
from typing import Any, Dict, List

class ReadingOrderBuilder:

    def __init__(self, x_tolerance: float = 30.0):
        self.x_tolerance = x_tolerance

    def _detect_columns(self, nodes: List[Dict[str, Any]]) -> int:
        if len(nodes) <= 1:
            return 1
        xs = sorted([n["bbox"]["x"] for n in nodes])
        groups = []
        for x in xs:
            if not groups:
                groups.append([x]); continue
            if abs(x-groups[-1][0]) <= self.x_tolerance:
                groups[-1].append(x)
            else:
                groups.append([x])
        return max(1, len(groups))

    def _sort_single_column(self, nodes):
        return sorted(nodes, key=lambda n:(n["bbox"]["y"], n["bbox"]["x"]))

    def _sort_multi_column(self, nodes):
        cols={}
        for n in nodes:
            x=n["bbox"]["x"]
            found=False
            for k in list(cols.keys()):
                if abs(x-k)<=self.x_tolerance:
                    cols[k].append(n)
                    found=True
                    break
            if not found:
                cols[x]=[n]
        ordered=[]
        for k in sorted(cols.keys()):
            ordered.extend(sorted(cols[k], key=lambda n:n["bbox"]["y"]))
        return ordered

    def build(self, page_layout: Dict[str, Any]) -> Dict[str, Any]:
        tree=page_layout["layout_tree"]
        nodes=tree["children"]
        column_count=self._detect_columns(nodes)

        ordered=self._sort_single_column(nodes) if column_count==1 else self._sort_multi_column(nodes)

        reading_order=[]
        graph_nodes=[]
        graph_edges=[]
        prev=None

        for i,node in enumerate(ordered, start=1):
            node["reading_order"]=i
            reading_order.append(node["node_id"])
            graph_nodes.append({"id":node["node_id"],"order":i})
            if prev:
                graph_edges.append({"from":prev,"to":node["node_id"]})
            prev=node["node_id"]

        profile=page_layout.get("layout_profile",{})
        profile["column_count"]=column_count

        return {
            "metadata":page_layout["metadata"],
            "layout_profile":profile,
            "layout_tree":tree,
            "reading_order":reading_order,
            "reading_graph":{
                "nodes":graph_nodes,
                "edges":graph_edges
            }
        }


## 3.2. Table Boundary Detection

> Structure Reconstruction - Table Boundary Detection

Input:
    page_layout (after Layout Pre-Detection)

Output:
    page_layout enriched with logical_tables

This module DOES NOT perform OCR.
It reconstructs logical table structure from TABLE_REGION and child objects.



In [97]:
from __future__ import annotations

from typing import Dict, Any, List

class TableBoundaryDetector:

    def __init__(self, row_tolerance: float = 6.0,
                 col_tolerance: float = 8.0):
        self.row_tolerance = row_tolerance
        self.col_tolerance = col_tolerance

    def _cluster(self, values: List[float], tol: float):
        groups = []
        for v in sorted(values):
            if not groups:
                groups.append([v])
            elif abs(v - groups[-1][0]) <= tol:
                groups[-1].append(v)
            else:
                groups.append([v])
        return [sum(g)/len(g) for g in groups]

    def _build_table(self, region: Dict[str, Any]) -> Dict[str, Any]:

        children = region.get("children", [])

        xs = []
        ys = []

        for c in children:
            bbox = c.get("bbox")
            if bbox:
                xs.append(bbox["x"])
                ys.append(bbox["y"])

        rows = self._cluster(ys, self.row_tolerance)
        cols = self._cluster(xs, self.col_tolerance)

        return {
            "table_id": region["node_id"],
            "region_id": region["node_id"],
            "bbox": region["bbox"],
            "row_count": len(rows),
            "column_count": len(cols),
            "rows": [
                {
                    "row_index": i,
                    "y": y
                }
                for i, y in enumerate(rows)
            ],
            "columns": [
                {
                    "column_index": i,
                    "x": x
                }
                for i, x in enumerate(cols)
            ],
            "cells": [],
            "continued": False,
            "confidence": region.get("confidence", 1.0)
        }

    def build(self, page_structure: Dict[str, Any]) -> Dict[str, Any]:

        tree = page_structure["layout_tree"]

        logical_tables = []

        for region in tree["children"]:

            if region["node_type"] != "TABLE_REGION":
                continue

            logical_tables.append(
                self._build_table(region)
            )

        page_structure["logical_tables"] = logical_tables

        page_structure.setdefault("structure_profile", {})
        page_structure["structure_profile"]["table_count"] = len(logical_tables)

        return page_structure


## 3.3. Multi-page Table Detection

> Structure Reconstruction - Multi-page Table Detection

Merge logical tables spanning consecutive pages.

Input:
    List[page_structure]

Output:
    {
        "pages": [...],
        "logical_tables": [...]
    }



In [98]:

from __future__ import annotations
from typing import Any, Dict, List


class MultiPageTableDetector:

    def __init__(
        self,
        header_match_threshold: float = 0.8,
        edge_margin: float = 50.0,
    ):
        self.header_match_threshold = header_match_threshold
        self.edge_margin = edge_margin

    def _header_signature(self, table: Dict[str, Any]) -> str:
        """
        Placeholder signature.
        V2: build from OCR/header cells.
        """
        return f'{table.get("column_count",0)}'

    def _is_bottom_table(self, table: Dict[str, Any], page_height: float) -> bool:
        b = table["bbox"]
        return (b["y"] + b["height"]) >= (page_height - self.edge_margin)

    def _is_top_table(self, table: Dict[str, Any]) -> bool:
        return table["bbox"]["y"] <= self.edge_margin

    def _merge_tables(self, first: Dict[str, Any], second: Dict[str, Any]) -> Dict[str, Any]:

        merged = dict(first)

        merged["pages"] = first.get("pages", [first.get("page_number")])
        merged["pages"].append(second.get("page_number"))

        merged["continued"] = True

        merged["row_count"] = (
            first.get("row_count", 0)
            + second.get("row_count", 0)
        )

        merged["rows"] = first.get("rows", []) + second.get("rows", [])

        merged["cells"] = first.get("cells", []) + second.get("cells", [])

        return merged

    def build(self, pages: List[Dict[str, Any]]) -> Dict[str, Any]:

        unified_tables: List[Dict[str, Any]] = []

        previous = None

        for page in pages:

            page_number = page["metadata"]["page_number"]

            page_height = page.get(
                "page_size",
                {}
            ).get("height", 999999)

            tables = page.get("logical_tables", [])

            for table in tables:

                table["page_number"] = page_number

                if previous is None:
                    previous = table
                    continue

                same_header = (
                    self._header_signature(previous)
                    == self._header_signature(table)
                )

                continued = (
                    previous.get("column_count")
                    == table.get("column_count")
                )

                if (
                    same_header
                    and continued
                    and self._is_bottom_table(previous, page_height)
                    and self._is_top_table(table)
                ):

                    previous = self._merge_tables(previous, table)

                else:

                    unified_tables.append(previous)

                    previous = table

        if previous is not None:
            unified_tables.append(previous)

        return {
            "pages": pages,
            "logical_tables": unified_tables,
            "structure_profile": {
                "logical_table_count": len(unified_tables)
            }
        }


## 3.4. StructurePipeline

In [112]:
import json
import os

class StructurePipeline:

    def process_document(self, layout_document_file):

        with open(layout_document_file, "r", encoding="utf-8") as f:
            layout_document = json.load(f)

        base_dir = os.path.dirname(layout_document_file)

        pages = []

        for page_info in layout_document["pages"]:

            layout_file = os.path.join(
                base_dir,
                "page_layout",
                page_info["layout_file"]
            )

            with open(layout_file, "r", encoding="utf-8") as f:
                page_layout = json.load(f)

            page_layout = ReadingOrderBuilder().build(page_layout)
            page_layout = TableBoundaryDetector().build(page_layout)

            pages.append(page_layout)

        document_structure = MultiPageTableDetector().build(pages)

        return document_structure

    def save(self,
             document_structure,
             output_file):

        os.makedirs(
            os.path.dirname(output_file),
            exist_ok=True
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                document_structure,
                f,
                ensure_ascii=False,
                indent=4
            )

        print(f"Saved: {output_file}")

In [113]:
pipeline = StructurePipeline()

document_structure = pipeline.process_document(
    "output/layout_document.json"
)

pipeline.save(
    document_structure,
    "output/document_structure.json"
)

Saved: output/document_structure.json
